# Dental Oral Health Multi-Task Training (Thesis_Results.csv)

This notebook trains a patch-based multi-task model (MGI, OHI, GEI) and prepares PI rule-based evaluation utilities. It is configured for the real dataset schema in `Thesis_Results.csv` and enforces CUDA-only training.

## Section 1: Environment Setup
Install dependencies, import modules, set seeds, and enforce GPU usage.

In [ ]:
%pip install -q -r ../requirements_model.txt

import json
import logging
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import seaborn as sns
import torch

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('train_notebook')

def set_global_seed(seed: int) -> None:
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA GPU is required for this training notebook. '
        'Install CUDA-enabled PyTorch and rerun.'
    )

device = torch.device('cuda')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

gpu_props = torch.cuda.get_device_properties(0)
logger.info('GPU: %s', gpu_props.name)
logger.info('VRAM: %.2f GB', gpu_props.total_memory / (1024 ** 3))
logger.info('PyTorch: %s | CUDA: %s', torch.__version__, torch.version.cuda)
logger.info('cuDNN: %s', torch.backends.cudnn.version())

## Section 2: Configuration Block
All notebook values are sourced from this single config dictionary.

In [ ]:
CONFIG = {
    'image_size': 512,
    'patch_size': 224,
    'batch_size': 8,
    'num_epochs': 50,
    'lr_backbone': 1e-5,
    'lr_heads': 1e-3,
    'dropout': 0.4,
    'k_folds': 5,
    'freeze_epochs': 10,
    'early_stopping_patience': 8,
    'num_classes_mgi': 5,
    'num_classes_ohi': 4,
    'num_classes_gei': 4,
    'backbone_model': 'vit_small_patch14_dinov2.lvd142m',
    'data_dir': '../Thesis_Data',
    'patch_dir': '../data/patches',
    'checkpoint_dir': '../models/checkpoints',
    'final_model_path': '../models/multitask_model.pth',
    'labels_file': '../Thesis_Data/Thesis_Results.csv',
    'seed': 42,
    'num_workers': 2,
    'yolo_model_name': 'yolov8n.pt',
    'yolo_weights_path': None,
    'sam_checkpoint_path': None,
    'plots_dir': '../outputs/plots',
    'onnx_path': '../models/oral_health_model.onnx'
}

set_global_seed(int(CONFIG['seed']))

for required_dir in [CONFIG['patch_dir'], CONFIG['checkpoint_dir'], CONFIG['plots_dir']]:
    Path(required_dir).mkdir(parents=True, exist_ok=True)

logger.info('Configuration loaded successfully.')

## Section 3: Dataset Loading and Label Parsing
Load records from `Thesis_Results.csv`, parse labels, and group by patient triplets.

In [ ]:
from training.dataset import PatientDataset, load_dataset

records = load_dataset(CONFIG['data_dir'])
patient_dataset = PatientDataset(records, require_complete_views=True, load_images=False)

patient_rows = []
for sample in patient_dataset:
    patient_rows.append({
        'patient_id': sample['patient_id'],
        'mgi': sample['labels']['mgi'],
        'ohi': sample['labels']['ohi'],
        'gei': sample['labels']['gei'],
    })

patient_df = pd.DataFrame(patient_rows).drop_duplicates('patient_id').reset_index(drop=True)

if patient_df.empty:
    raise RuntimeError('No complete patient triplets were found. Check data paths and naming.')

CONFIG['num_classes_mgi'] = int(patient_df['mgi'].nunique())
CONFIG['num_classes_ohi'] = int(patient_df['ohi'].nunique())
CONFIG['num_classes_gei'] = int(patient_df['gei'].nunique())

logger.info('Loaded %d image-level records', len(records))
logger.info('Loaded %d patient-level complete triplets', len(patient_df))
logger.info('Class counts -> MGI:%d OHI:%d GEI:%d', CONFIG['num_classes_mgi'], CONFIG['num_classes_ohi'], CONFIG['num_classes_gei'])

## Section 4: Preprocessing Pipeline
Run standardization, YOLO detection, SAM segmentation, and patch generation.

In [ ]:
from preprocessing.patch_generator import preprocess_all_images
from preprocessing.sam_segmentation import GumSegmentor
from preprocessing.yolo_detection import ToothDetector

detector = ToothDetector(
    model_name=CONFIG['yolo_model_name'],
    weights_path=CONFIG['yolo_weights_path'],
    device='cuda'
)
segmentor = GumSegmentor(
    checkpoint_path=CONFIG['sam_checkpoint_path'],
    device='cuda'
)

patch_df = preprocess_all_images(
    dataset=records,
    detector=detector,
    segmentor=segmentor,
    patch_dir=CONFIG['patch_dir'],
    image_size=int(CONFIG['image_size']),
    patch_size=int(CONFIG['patch_size']),
)

if patch_df.empty:
    raise RuntimeError('Patch generation produced zero samples. Please inspect preprocessing logs.')

logger.info('Generated %d patches', len(patch_df))
patch_df.head()

## Section 5: Data Augmentation
Define Albumentations training and validation pipelines.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.GaussianBlur(blur_limit=3, p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
    A.CLAHE(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

logger.info('Augmentation pipelines prepared.')

## Section 6: PyTorch Dataset and DataLoader
Build patch-level train/validation data loaders.

In [ ]:
from sklearn.model_selection import train_test_split
from training.dataset import get_dataloaders

patch_df['patient_id'] = patch_df['patient_id'].astype(str)
patient_label_df = patch_df.groupby('patient_id', as_index=False).agg(mgi=('mgi', 'first'))
train_pat, val_pat = train_test_split(
    patient_label_df,
    test_size=0.2,
    random_state=int(CONFIG['seed']),
    stratify=patient_label_df['mgi']
)
train_pat_set = set(train_pat['patient_id'].tolist())
val_pat_set = set(val_pat['patient_id'].tolist())

train_indices = patch_df.index[patch_df['patient_id'].isin(train_pat_set)].tolist()
val_indices = patch_df.index[patch_df['patient_id'].isin(val_pat_set)].tolist()

train_loader, val_loader = get_dataloaders(
    fold_train_indices=train_indices,
    fold_val_indices=val_indices,
    patch_df=patch_df,
    batch_size=int(CONFIG['batch_size']),
    num_workers=int(CONFIG['num_workers']),
    train_transform=train_transform,
    val_transform=val_transform,
)

sample_images, sample_targets = next(iter(train_loader))
logger.info('Sample batch shape: %s', tuple(sample_images.shape))
logger.info('Sample target keys: %s', list(sample_targets.keys()))

## Section 7: Multi-Task Model
Instantiate the DINOv2 multitask architecture and verify backbone freezing methods.

In [ ]:
from training.model import OralHealthModel

model = OralHealthModel(CONFIG).to(device)
model.freeze_backbone()
model.unfreeze_last_n_blocks(4)

assert next(model.parameters()).is_cuda, 'Model parameters are not on CUDA.'
logger.info('Model initialized on CUDA with backbone %s', CONFIG['backbone_model'])

## Section 8: Loss Function
Initialize uncertainty-weighted multi-task cross-entropy loss.

In [ ]:
from training.loss import MultiTaskLoss

loss_fn = MultiTaskLoss().to(device)
logger.info('MultiTaskLoss initialized with learnable task uncertainties.')

## Section 9: Training Loop
Import train/eval routines and early stopping utility.

In [ ]:
from training.trainer import EarlyStopping, evaluate, train_kfold, train_one_epoch

logger.info('Training helpers imported: train_one_epoch, evaluate, EarlyStopping, train_kfold')

## Section 10: K-Fold Training
Train with patient-level stratified K-fold and save best checkpoints.

In [ ]:
training_results = train_kfold(
    patch_df=patch_df,
    config=CONFIG,
    train_transform=train_transform,
    val_transform=val_transform,
)

history_path = Path(CONFIG['checkpoint_dir']) / 'training_history.json'
with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(training_results, f, indent=2, default=str)

logger.info('K-fold training completed. History saved to %s', history_path)
logger.info('Average macro-F1: %s', training_results['avg_val_f1'])

## Section 11: Visualization and Evaluation
Plot fold loss curves, confusion matrices, and classification reports.

In [ ]:
from sklearn.metrics import classification_report

plots_dir = Path(CONFIG['plots_dir'])
plots_dir.mkdir(parents=True, exist_ok=True)

# Loss curves
for fold_idx, history in enumerate(training_results['histories'], start=1):
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_total_loss'] for row in history]
    val_loss = [row['val_loss'] for row in history]

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_loss, label='Train Loss')
    plt.plot(epochs, val_loss, label='Val Loss')
    plt.axvline(int(CONFIG['freeze_epochs']), linestyle='--', color='gray', label='Unfreeze point')
    plt.title(f'Fold {fold_idx} Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    loss_plot_path = plots_dir / f'fold_{fold_idx}_loss_curve.png'
    plt.tight_layout()
    plt.savefig(loss_plot_path, dpi=150)
    plt.close()

# Best fold by average macro-F1
def avg_f1(summary):
    vals = summary['final_val_f1']
    return (vals['mgi'] + vals['ohi'] + vals['gei']) / 3.0

best_fold_summary = max(training_results['fold_summaries'], key=avg_f1)
best_fold = int(best_fold_summary['fold'])
logger.info('Best fold by macro-F1 average: %d', best_fold)

for task in ('mgi', 'ohi', 'gei'):
    cm = np.array(best_fold_summary['confusion_matrices'][task])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{task.upper()} Confusion Matrix (Fold {best_fold})')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    cm_path = plots_dir / f'best_fold_{best_fold}_{task}_confusion_matrix.png'
    plt.tight_layout()
    plt.savefig(cm_path, dpi=150)
    plt.close()

    report = classification_report(
        best_fold_summary['y_true'][task],
        best_fold_summary['y_pred'][task],
        zero_division=0
    )
    logger.info('\n%s classification report (best fold %d):\n%s', task.upper(), best_fold, report)
    report_path = plots_dir / f'best_fold_{best_fold}_{task}_classification_report.txt'
    report_path.write_text(report, encoding='utf-8')

logger.info('Saved all evaluation plots/reports to %s', plots_dir)

## Section 12: Export Final Model
Load best fold checkpoint, save final state dict, export ONNX, and run ONNX smoke test.

In [ ]:
best_fold_ckpt = Path(CONFIG['checkpoint_dir']) / f"checkpoint_fold_{best_fold}.pth"
final_model_path = Path(CONFIG['final_model_path'])
onnx_path = Path(CONFIG['onnx_path'])

best_model = OralHealthModel(CONFIG).to(device)
ckpt = torch.load(best_fold_ckpt, map_location=device)
best_model.load_state_dict(ckpt['model_state_dict'], strict=False)
best_model.eval()

final_model_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(best_model.state_dict(), final_model_path)
logger.info('Saved final model state_dict to %s', final_model_path)

class ONNXExportWrapper(torch.nn.Module):
    """Wrap dict-output model into tuple-output model for ONNX export."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        return out['mgi'], out['ohi'], out['gei']

export_model = ONNXExportWrapper(best_model).to(device)
dummy_input = torch.randn(1, 3, int(CONFIG['patch_size']), int(CONFIG['patch_size']), device=device)

onnx_path.parent.mkdir(parents=True, exist_ok=True)
torch.onnx.export(
    export_model,
    dummy_input,
    str(onnx_path),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input_patch'],
    output_names=['mgi_logits', 'ohi_logits', 'gei_logits'],
)
logger.info('Exported ONNX model to %s', onnx_path)

session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
ort_inputs = {'input_patch': np.random.randn(1, 3, int(CONFIG['patch_size']), int(CONFIG['patch_size'])).astype(np.float32)}
ort_outputs = session.run(None, ort_inputs)
logger.info('ONNX smoke test outputs: %s', [o.shape for o in ort_outputs])

## Section 13: Plaque Index Algorithm
Run the plaque index function on a sample patch.

In [ ]:
from inference.plaque_index import compute_plaque_index, plaque_score_to_label

sample_patch_path = Path(patch_df.iloc[0]['patch_path'])
sample_patch_bgr = cv2.imread(str(sample_patch_path), cv2.IMREAD_COLOR)
if sample_patch_bgr is None:
    raise RuntimeError(f'Failed to load sample patch: {sample_patch_path}')
sample_patch_rgb = cv2.cvtColor(sample_patch_bgr, cv2.COLOR_BGR2RGB)
sample_mask = np.ones(sample_patch_rgb.shape[:2], dtype=np.uint8)

pi_ratio, pi_score = compute_plaque_index(sample_patch_rgb, sample_mask)
pi_label = plaque_score_to_label(pi_score)
logger.info('Sample patch PI -> ratio=%.4f score=%d label=%s', pi_ratio, pi_score, pi_label)